# IndicTranslate — Inference Walkthrough

IndicTranslate is Gemma-4-E4B instruction-tuned for Indic translation: 25 language-script
combinations, 44 directions. It is a decoder-only LLM, not a seq2seq model — there are no language
tokens and no `forced_bos_token_id`. The target language is an English name interpolated into an
instruction sentence, and **the source language is never named**.

| path | class | use it for |
|---|---|---|
| offline, in-process | `IndicMTEngine(backend="vllm")` | throughput — a whole corpus as one continuous batch |
| offline, in-process | `IndicMTEngine(backend="hf")` | a reference run, or an **unmerged** PEFT adapter |
| a running server | `MTClient` | stock `vllm serve`, OpenAI-compatible |

All three go through the same `build_conversation()`, so no two runtimes can drift apart. Do not
hand-write a `messages` payload — see [docs/mt/prompt_contract.md](../../docs/mt/prompt_contract.md).

**Prerequisites**

- **GPU required** for sections 1–5. Section 6 needs only a network route to a server.
- MT installs into **its own environment** — it needs `transformers>=5.12` and `vllm>=0.20` for the
  Gemma 4 architecture, which the TTS pins cannot satisfy:
  `./install.sh && source .venv/bin/activate`
- **The checkpoint `bodhan-ai/indic-translate` is public.** No authentication needed
  (`hf auth login`, or `export HF_TOKEN=...`) or point `MODEL` at a local directory. First load
  pulls ~15.9 GB.
- The failure mode to internalise: **a broken prompt does not raise.** The model returns fluent,
  plausible text that is simply worse, and nothing in the logs says so.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import torch

import bodhan_genai.mt

print("bodhan-genai:", bodhan_genai.mt.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
for pkg in ("transformers", "vllm"):  # MT floors: transformers>=5.12, vllm>=0.20
    try:
        print(f"{pkg}:", version(pkg))
    except PackageNotFoundError:
        print(f"{pkg}: not installed")

# A vLLM wheel built for a newer CUDA than the driver imports cleanly and then silently
# reports no GPU, so assert rather than discover it 16 GB into a model load.
assert torch.cuda.is_available(), "inference needs a GPU node (or the wrong venv is active)"
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
# --- edit these ------------------------------------------------------------
MODEL = "bodhan-ai/indic-translate"  # gated Hub repo, or /path/to/merged/checkpoint
TGT_LANG = "hin_Deva"  # FLORES code, bare name ("Hindi"), or qualified name — see §4
SERVER_URL = "http://localhost:8000/v1"  # only used in §6
# ----------------------------------------------------------------------------

SENTENCE = "The committee approved the proposal after a long debate."
SEGMENTS = [
    "The meeting has been postponed to next Tuesday.",
    "Applications close at the end of the month.",
    "The report has not been submitted yet.",
    "समिति ने प्रस्ताव को मंजूरी दे दी।",  # same call handles the reverse direction
]

## 1. Offline translation

`IndicMTEngine` loads the model once and translates on demand. Construction takes tens of seconds
(weights are ~15.9 GB); every `translate` after that is fast, so build the engine **once per
process**. It is a context manager, and `close()` is idempotent.

`max_model_len` defaults to 32768 — the only window measured end to end. Sentence workloads do not
need it, and a smaller window leaves KV-cache room for a bigger batch.

In [ ]:
from bodhan_genai.mt import IndicMTEngine

engine = IndicMTEngine(
    MODEL,
    # backend="vllm" is the default; see §5 for the HF backend.
    max_model_len=8192,  # 32768 is the default; sentences do not need the window
    gpu_memory_utilization=0.90,
    # enforce_eager=True is the default: it is the configuration the published scores
    # were measured with, and CUDA-graph capture buys little at translation lengths.
)

In [ ]:
result = engine.translate(SENTENCE, tgt_lang=TGT_LANG)

# Note what was never passed: a SOURCE language. The prompt names only the target and
# the model infers the source — which is why one call handles both directions.
print("source           :", result.source)
print(f"{result.tgt_lang:<17}:", result.text)
print(f"[{result.gen_time_s:.2f}s, ok={result.ok}]")

back = engine.translate(result.text, tgt_lang="English")
print("back-translation :", back.text)

# Generation failure never raises — it lands in .error, so one bad row cannot sink a run.
if not result.ok:
    print("FAILED:", result.error)

## 2. Sampling control — the default is greedy, and that matters

`MTSamplingConfig` defaults to `temperature=0.0`, i.e. **greedy decoding**. That is deliberate and
it is the recommendation for translation:

- A sampled translation is a *different translation every time*. Two runs of the same eval then
  differ for reasons that have nothing to do with the model, and a regression becomes unmeasurable.
- Greedy is the setting the published scores were measured with. Raise the temperature only if you
  want variety and are prepared for the numbers to move.

`top_p` and `seed` are only meaningful when `temperature > 0`; under greedy decoding the backends
drop them rather than sending contradictory parameters downstream.

Set engine-wide defaults at construction (`IndicMTEngine(..., sampling=MTSamplingConfig(...))`) or
override per call. Per-call kwargs go through `merged()`: `None` falls through to the engine
default, and an unknown key raises `TypeError` instead of being silently ignored.

**Sizing `max_new_tokens`**: 512 suits sentences, ~2048 a paragraph, ~8192 a document. Indic
targets need roughly 1.5–2× the English source token count — under-budget it and you truncate
mid-sentence.

In [ ]:
from bodhan_genai.mt import MTSamplingConfig

sc = MTSamplingConfig()
sampled = sc.merged(temperature=0.7)
print("canonical defaults:", sc)
print("greedy?", sc.greedy)
print("merged(temperature=0.7):", sampled, "-> greedy?", sampled.greedy)
print("a None override falls through:", sc.merged(temperature=None))

# Greedy is reproducible: the same input gives the same output, run to run.
a = engine.translate(SENTENCE, tgt_lang=TGT_LANG).text
b = engine.translate(SENTENCE, tgt_lang=TGT_LANG).text
print("\ngreedy run 1 == run 2 :", a == b)

# Sampled is not. Useful for generating variants; useless for measuring a change.
s1 = engine.translate(SENTENCE, tgt_lang=TGT_LANG, temperature=0.9, top_p=0.9, seed=1).text
s2 = engine.translate(SENTENCE, tgt_lang=TGT_LANG, temperature=0.9, top_p=0.9, seed=2).text
print("sampled seed1 == seed2:", s1 == s2)
print(" ", s1)
print(" ", s2)

# engine.translate(SENTENCE, tgt_lang=TGT_LANG, top_k=50)   # TypeError: unknown field

## 3. Batch translation

`translate_batch` hands the whole list to vLLM as **one continuous batch**. Chunking it yourself
just idles the GPU between chunks. The target language is resolved once, before anything is
generated, so a typo fails immediately rather than after a full batch.

Per-row failures land in `result.error` instead of raising, and results stay aligned with the
input. `src_lang` is bookkeeping only — it is carried into `as_record()` and never enters the
prompt.

In [ ]:
import json

results = engine.translate_batch(SEGMENTS, tgt_lang=TGT_LANG, src_lang="eng_Latn")

for i, r in enumerate(results):
    if r.ok:
        print(f"[{i}] {r.text}")
    else:
        print(f"[{i}] FAILED: {r.error}")

# gen_time_s on a batched call is the per-request average (wall clock / batch size),
# not an independent measurement per row.
failed = sum(1 for r in results if not r.ok)
total_s = results[0].gen_time_s * len(results)
print(f"\n{len(results)} segments, {failed} failed, {total_s:.2f}s total")

# as_record() is the JSONL schema the shipped CLIs write.
print(json.dumps(results[0].as_record(), ensure_ascii=False, indent=2))

# prompt_tokens / generated_tokens stay 0 here: the offline engine does not count
# tokens. They are filled in by MTClient, from the server's usage block (§6).

In [ ]:
# A whole document as ONE request, so structure and cross-sentence context survive.
# Raise max_new_tokens accordingly — the default of 512 is sized for sentences and
# will truncate. The document also has to fit inside max_model_len (8192 above).
DOCUMENT = """The state government has announced a new scholarship scheme for students
from rural districts. Applications open on the first of next month and close six weeks
later. Candidates must provide proof of residence and their previous year's marksheet.

The scheme covers tuition fees in full and provides a monthly stipend for living costs."""

doc = engine.translate_document(DOCUMENT, tgt_lang=TGT_LANG, max_new_tokens=2048)
print(doc.text)

## 4. Language selection — and the multi-script trap

`tgt_lang` accepts three equivalent forms, all resolved through `resolve_language()`:

| form | example |
|---|---|
| FLORES-200 style code | `hin_Deva` |
| bare name | `hindi`, `Hindi` |
| qualified name | `Manipuri (Bengali script)` |

**This is the cell to read twice.** Three languages are carried in more than one script, and the
parenthetical in the prompt is **functional, not decoration** — it is what selects the output
script:

| language | scripts | a bare name resolves to |
|---|---|---|
| Kashmiri | `kas_Arab` (Perso-Arabic) | `kas_Arab` |
| Manipuri | `mni_Mtei` (Meitei), `mni_Beng` (Bengali) | `mni_Mtei` |
| Sindhi | `snd_Deva` (Devanagari), `snd_Arab` (Perso-Arabic) | `snd_Deva` |

Measured: prompting bare `"Sindhi"` instead of `"Sindhi (Devanagari script)"` scored
**6.67 vs 36.57 chrF++ — a 29.9-point drop**. Bare `"Kashmiri"` costs about 0.5 BLEU. Both look
like a bad *language*, not a bad *prompt*, which is exactly why this is worth internalising.
`resolve_language()` protects you as long as you go through it: pass the explicit code (or the
fully qualified name) whenever you want a specific script, and never build the instruction by
hand. Full detail in [docs/mt/prompt_contract.md](../../docs/mt/prompt_contract.md).

In [ ]:
from bodhan_genai.mt import LANGUAGE_NAMES, build_conversation, build_instruction, resolve_language
from bodhan_genai.mt.templates import DEFAULT_SCRIPT

print(f"{len(LANGUAGE_NAMES)} language-script targets (22 Eighth-Schedule languages + English):")
width = max(len(code) for code in LANGUAGE_NAMES)
for code, name in LANGUAGE_NAMES.items():
    print(f"  {code:<{width}}  {name}")

# Three input forms, one resolved prompt name.
for form in ("hin_Deva", "hindi", "Hindi"):
    print(f"\nresolve_language({form!r}) -> {resolve_language(form)!r}")

# The multi-script languages, and what a BARE name silently resolves to.
print("\nbare name -> default script:")
for bare, code in DEFAULT_SCRIPT.items():
    print(f"  {bare:<10} -> {code}  ({LANGUAGE_NAMES[code]})")

# The instruction the model actually sees. This is the whole difference.
print("\n" + build_instruction("The meeting was postponed.", "Sindhi"))
print("\n" + build_instruction("The meeting was postponed.", "snd_Arab"))

# And the full request payload: exactly one user turn, no system turn.
payload = build_conversation("The meeting was postponed.", "snd_Arab")
print("\n" + json.dumps(payload, ensure_ascii=False, indent=2))

In [ ]:
# The qualifier changes the output script, not just the label.
for code in ("snd_Deva", "snd_Arab", "mni_Mtei", "mni_Beng"):
    r = engine.translate("The meeting was postponed.", tgt_lang=code)
    print(f"{code}  {r.tgt_lang:<32} {r.text}")

# A typo fails fast, before any generation happens.
try:
    engine.translate("Hello", tgt_lang="Hindustani")
except ValueError as exc:
    print("\n" + str(exc).splitlines()[0])

## 5. HF backend — PEFT adapters and debugging

`backend="hf"` swaps vLLM for `AutoModelForImageTextToText` + `generate()`. Slower for batches, but
it loads an **unmerged** LoRA adapter directly through `adapter_dir` and is easier to step through.

`adapter_dir` with `backend="vllm"` raises: vLLM wants one self-contained directory, so merge first
(`python -m bodhan_genai.mt.training.merge`).

**Close the vLLM engine before constructing the second one.** One engine per process is the rule —
both want most of the GPU, and vLLM has already reserved 90 % of it.

In [ ]:
engine.close()  # release the vLLM engine before loading another backend

hf_engine = IndicMTEngine(
    MODEL,
    backend="hf",
    attn_implementation="sdpa",
    device="auto",
    # adapter_dir="training_output/mt-lora-8k",   # <- unmerged adapter from notebooks/mt/training.ipynb
)
print(hf_engine.translate(SENTENCE, tgt_lang=TGT_LANG).text)
hf_engine.close()

# For reference, the error you get if you try the adapter on the vLLM backend:
try:
    IndicMTEngine(MODEL, adapter_dir="/some/adapter")
except ValueError as exc:
    print("\n", exc)

## 6. Talking to a running server

There is no custom server in this repo: IndicTranslate runs on **stock `vllm serve`**, speaking the
OpenAI chat API. `scripts/mt/serve.sh` is a wrapper that picks a free port, waits for readiness,
and writes `vllm-serve.info` with the port it actually bound.

```bash
scripts/mt/serve.sh                                   # published checkpoint, GPU 0, port 8000
CHECKPOINT=merged-ckpt-4400 scripts/mt/serve.sh       # your own finetune
GPU=3 PORT=8100 MAX_MODEL_LEN=32768 scripts/mt/serve.sh
```

You could curl it. What `MTClient` adds is the prompt contract — a hand-rolled payload that names
the source language, or adds a system turn, still returns fluent text, just measurably worse text.
`health()` checks the *served model name*, not just that something answered: on a shared box the
port may belong to someone else's server.

**The client side needs no GPU** — the server holds the model. (The assert in the setup cell is
about §1–5.)

In [ ]:
from bodhan_genai.mt.serving import MTClient

client = MTClient(SERVER_URL, model="indic_translate")

if not client.health():
    print(f"nothing serving {client.model!r} at {SERVER_URL}; start scripts/mt/serve.sh")
else:
    r = client.translate(SENTENCE, tgt_lang=TGT_LANG)
    print(r.text)
    # Unlike the offline engine, the server reports usage, so these are populated.
    print(f"prompt {r.prompt_tokens} tok | generated {r.generated_tokens} tok")

    # The server batches internally; num_workers only keeps its queue fed. Input order
    # is preserved regardless of completion order, so a failed row never shifts the rest.
    batch = client.translate_batch(SEGMENTS, tgt_lang="Tamil", num_workers=16)
    for r in batch:
        print(r.text if r.ok else f"[error] {r.error}")

## Where to go next

- **Bulk files** (a corpus, several targets, one resident engine):
  `scripts/mt/infer.sh --tgt-lang mar_Deva --input-file segments.txt --output-file out.jsonl`,
  defaults in `configs/mt/infer/offline_vllm.yaml`. Add `--document --max-new-tokens 8192` to treat
  the input as one document. Standalone scripts: `examples/mt/basic_translate.py`,
  `examples/mt/batch_vllm.py`, `examples/mt/serve_client.py`.
- **List the languages from a shell**: `python -m bodhan_genai.mt.inference.cli hf --list-languages`.
- **Score a checkpoint**: `scripts/mt/eval.sh --langs hin_Deva --directions en-xx --max-samples 32`
  for a smoke run, bare `scripts/mt/eval.sh` for full IN22. The noise floor is about ±0.06 chrF++,
  so treat anything inside ±0.2 as a tie — [docs/mt/eval.md](../../docs/mt/eval.md).
- **Train your own checkpoint**: [notebooks/mt/training.ipynb](training.ipynb).
- **Everything on one page**: [docs/mt/end-to-end.md](../../docs/mt/end-to-end.md).